# Phlox 0.8B fine-tune — QLoRA on Qwen3.5-0.8B

**Before Run-all**: upload to `/content/`: `sft_train.jsonl`, `sft_dev.jsonl`, `eval.py` (from `training/`).

Pipeline: QLoRA (r=16) → merge → eval vs base → ONNX export (+q8) → zip for download.
Runtime on free T4: ~45–70 min train, ~30 min evals, ~15 min export.

In [ ]:
!pip install -q -U transformers peft bitsandbytes accelerate datasets onnx onnxruntime optimum
!pip install -q "flash-linear-attention==0.4.2" causal-conv1d  # NOT 0.5.x (triton parser clash); T4 may still fall back (pre-Ampere)
!nvidia-smi -L

In [ ]:
import json, torch
from transformers import AutoProcessor, AutoTokenizer

BASE = "Qwen/Qwen3.5-0.8B"
MAX_LEN = 3072

tok = AutoTokenizer.from_pretrained(BASE)

def render_prompt(messages):
    try:
        return tok.apply_chat_template(messages, add_generation_prompt=True, enable_thinking=False, tokenize=False)
    except TypeError:
        return tok.apply_chat_template(messages, add_generation_prompt=True, tokenize=False)

def load_examples(path):
    rows, dropped = [], 0
    for line in open(path):
        ex = json.loads(line)
        prompt_ids = tok(render_prompt(ex["prompt"]), add_special_tokens=False)["input_ids"]
        comp_ids = tok(ex["completion"], add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
        if len(prompt_ids) + len(comp_ids) > MAX_LEN:
            dropped += 1
            continue
        rows.append({"input_ids": prompt_ids + comp_ids, "labels": [-100] * len(prompt_ids) + comp_ids})
    print(f"{path}: {len(rows)} kept, {dropped} dropped (> {MAX_LEN} tokens)")
    return rows

train_rows = load_examples("/content/sft_train.jsonl")
dev_rows = load_examples("/content/sft_dev.jsonl")

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map="auto")

# ponytail: if this base turns out to be VL, freeze the vision tower — we train text side only
for name, p in model.named_parameters():
    if "visual" in name.lower() or "vision" in name.lower():
        p.requires_grad_(False)

lora = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
)
model = get_peft_model(model, lora)
model.print_trainable_parameters()

In [ ]:
import torch
from torch.utils.data import Dataset
from transformers import Trainer, TrainingArguments

class SFT(Dataset):
    def __init__(self, rows): self.rows = rows
    def __len__(self): return len(self.rows)
    def __getitem__(self, i): return self.rows[i]

def collate(batch):
    maxlen = max(len(b["input_ids"]) for b in batch)
    pad = tok.eos_token_id
    input_ids, labels, attn = [], [], []
    for b in batch:
        n = len(b["input_ids"])
        input_ids.append(b["input_ids"] + [pad] * (maxlen - n))
        labels.append(b["labels"] + [-100] * (maxlen - n))
        attn.append([1] * n + [0] * (maxlen - n))
    return {"input_ids": torch.tensor(input_ids), "labels": torch.tensor(labels),
            "attention_mask": torch.tensor(attn)}

args = TrainingArguments(
    output_dir="/content/ckpt", per_device_train_batch_size=8, gradient_accumulation_steps=2,
    num_train_epochs=2, learning_rate=1e-4, lr_scheduler_type="cosine",
    logging_steps=50, save_strategy="no", report_to=[], fp16=True, bf16=False,
    gradient_checkpointing=True, dataloader_num_workers=2, seed=42,
)
trainer = Trainer(model=model, args=args, train_dataset=SFT(train_rows), data_collator=collate)
trainer.train()

In [ ]:
# Merge LoRA -> full weights (fp16) for eval + export
merged = model.merge_and_unload()
merged = merged.to(torch.float16)
merged.save_pretrained("/content/qwen-phlox-merged")
tok.save_pretrained("/content/qwen-phlox-merged")
print("merged saved")

In [ ]:
# Eval gate: base vs fine-tuned (dev subset keeps this under ~30 min on T4)
!python eval.py --model {BASE} --dev /content/sft_dev.jsonl --limit 250 --out /content/eval_base.json || true
!python eval.py --model /content/qwen-phlox-merged --dev /content/sft_dev.jsonl --limit 250 --out /content/eval_tuned.json

In [ ]:
from google.colab import files as gfiles
!python -m optimum.exporters.onnx --model /content/qwen-phlox-merged --task text-generation-with-past /content/onnx/phlox-0.8b
# q8 for the wasm fallback path
from onnxruntime.quantization import quantize_dynamic, QuantType
import glob, shutil, os
os.makedirs("/content/onnx/phlox-0.8b-q8", exist_ok=True)
for f in glob.glob("/content/onnx/phlox-0.8b/*.onnx"):
    quantize_dynamic(f, f.replace("/phlox-0.8b/", "/phlox-0.8b-q8/"), weight_type=QuantType.QInt8)
!cd /content/onnx && zip -rq /content/phlox-0.8b-onnx.zip phlox-0.8b phlox-0.8b-q8
print("download phlox-0.8b-onnx.zip (+ eval_*.json) — q4/webgpu packaging happens repo-side")